In [ ]:
from src.data_processing.data_loader import MovieLensDataLoader
import pandas as pd
loader = MovieLensDataLoader()
data_dict = loader.load_data()

print("Dobijamo metapodatke iz API...")

await loader.letterboxd_data_async(max_concurrent_requests=100)

print(f"\n[Easy] Dobijeno ovoliko filmova: {len(loader.movie_data)}")

if loader.movie_data:
    print("\nPrimer prvog filma:")
    import pprint
    pprint.pprint(loader.movie_data[0])

In [ ]:
import pandas as pd
movie_metadata = loader.movie_data
movie_metadata
df = pd.DataFrame(movie_metadata)

def cleaner(x):
    if isinstance(x, list):
        return [str.lower(i).replace(' ', '-') for i in x]
    elif isinstance(x, str):
        return str.lower(x).replace(' ', '-')
    return ""
df.head()
df['title_clean'] = df['title'].apply(cleaner)
df['main_actor_clean'] = df['main_actor'].apply(cleaner)
df['director_clean'] = df['director'].apply(cleaner)
df["rating"] = df["rating"].astype(float) / 2.0
df["cast_clean"] = df["cast"].apply(cleaner).apply(lambda x: [i for i in x if i])
a = loader.preprocess_movies()
df['genres_text'] = a.apply(lambda row: ' '.join([col.lower().replace(" ", "") for col in a.columns if row[col]]), 
    axis=1)
#df["genres_clean"] = df["genres"].apply(cleaner).apply(lambda x: [i for i in x if i])

df.head(10)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

median_runtime = df["runtime"].replace(0, np.nan).median()
df["runtime_filled"] = df["runtime"].replace({0: median_runtime, np.nan: median_runtime})

df["runtime_log"] = np.log1p(df["runtime_filled"])

scaler = MinMaxScaler()
df["runtime_scaled"] = scaler.fit_transform(df[["runtime_log"]])
df[["runtime", "runtime_filled", "runtime_log", "runtime_scaled"]].sort_values("runtime_scaled").head(50)

In [ ]:
df['main_actor_rating'] = df.groupby('main_actor_clean')['rating'].transform('mean')
median_rating_actor = df["main_actor_rating"].replace(0, np.nan).median()
df["main_actor_rating_filled"] = df["main_actor_rating"].replace({0: median_rating_actor, np.nan: median_rating_actor})

df["main_actor_rating_log"] = np.log1p(df["main_actor_rating_filled"])

scaler = MinMaxScaler()
df["main_actor_rating_scaled"] = scaler.fit_transform(df[["main_actor_rating_log"]])
df.head(10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# --- 1. Подготовка данных (Заполнение пропусков) ---
median_runtime = df["runtime"].replace(0, np.nan).median()
df["runtime_filled"] = df["runtime"].replace(
    {0: median_runtime, np.nan: median_runtime}
)

# --- 2. Клиппинг (Срезаем топ-1% экстремально длинных и коротких фильмов) ---
lower_bound = df["runtime_filled"].quantile(0.01)  # Отрежет скрытый мусор
upper_bound = df["runtime_filled"].quantile(0.99)  # Отрежет фильмы длиннее ~3.5 часов

df["runtime_clipped"] = df["runtime_filled"].clip(lower_bound, upper_bound)

# --- 3. Масштабирование в диапазон [0, 1] ---
scaler = MinMaxScaler()
df["runtime_scaled"] = scaler.fit_transform(df[["runtime_clipped"]])

# --- 4. Отрисовка новых графиков в Jupyter ---
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# График 1: Исходные данные с хвостами
sns.histplot(
    df["runtime_filled"], bins=50, kde=True, ax=axes[0], color="royalblue"
)
axes[0].set_title(
    f"1. Исходная длительность\n(Хвост уходит за {int(df['runtime_filled'].max())} мин)",
    fontsize=11,
)
axes[0].set_xlabel("Минуты")
axes[0].set_ylabel("Количество фильмов")

# График 2: Данные после клиппинга
sns.histplot(
    df["runtime_clipped"], bins=50, kde=True, ax=axes[1], color="teal"
)
axes[1].set_title(
    f"2. После клиппинга (1%-99%)\n(Ограничили: {int(lower_bound)} - {int(upper_bound)} мин)",
    fontsize=11,
)
axes[1].set_xlabel("Минуты (без аномалий)")
axes[1].set_ylabel("")

# График 3: Финальный признак для Content-Based
sns.histplot(
    df["runtime_scaled"], bins=50, kde=True, ax=axes[2], color="darkorange"
)
axes[2].set_title(
    "3. Итоговый признак для ML\n(Растянут от 0 до 1)", fontsize=11
)
axes[2].set_xlabel("Масштабированное значение")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer

# 1. Заполняем пропуски медианой (если они есть)
median_year = df["year"].astype(int).median()
df["year_filled"] = df["year"].fillna(median_year)

# 2. Применяем QuantileTransformer с нормальным распределением
# Обучаем трансформер
qt_year = QuantileTransformer(
    output_distribution="normal", n_quantiles=1000, random_state=42
)
year_transformed = qt_year.fit_transform(df[["year_filled"]])

# 3. Так как 'normal' выдает значения в пределах [-3, 3], сжимаем их в [0, 1] через MinMaxScaler
scaler_year = MinMaxScaler()
df["year_scaled"] = scaler_year.fit_transform(year_transformed)

# --- Визуализация ---
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(df["year_filled"], bins=40, kde=True, ax=axes[0], color="purple")
axes[0].set_title("1. Исходное распределение годов\n(Тяжелый хвост слева)")
axes[0].set_xlabel("Год")

sns.histplot(df["year_scaled"], bins=40, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("2. Итоговый признак для ML\n(Идеальная плавность от 0 до 1)")
axes[1].set_xlabel("Масштабированное значение")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tfidf_main_actor = TfidfVectorizer()
tfidf_main_actor_matrix = tfidf_main_actor.fit_transform(df["main_actor_clean"].fillna(""))
#print(tfidf_main_actor_matrix)
tfidf_director = TfidfVectorizer()
tfidf_director_matrix = tfidf_director.fit_transform(df["director_clean"].fillna(""))
print(tfidf_director_matrix)

In [ ]:
df['cast_joined'] = df['cast_clean'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))
tfidf_cast = TfidfVectorizer()
cast_matrix = tfidf_cast.fit_transform(df['cast_joined'])
print(cast_matrix)
df.head(10)


In [ ]:
from scipy.sparse import csr_matrix
numerical_features = df[['runtime_scaled', 'year_scaled', 'main_actor_rating_scaled']].values
numerical_matrix = csr_matrix(numerical_features)
print(numerical_matrix)